# Temporal Crosscoders — Toy Model

We test whether a **temporal crosscoder** — an SAE that sees a window of $T$ consecutive activation vectors rather than a single one — can better recover true underlying features when those features exhibit **temporal correlation**.

The ground-truth setup (50 orthogonal features in $\mathbb{R}^{100}$) mirrors [Chanin et al. (2025)](https://arxiv.org/abs/2408.05147) and the original `sae_token_embeddings` notebook.

## Architecture comparison

| Model | Input | Encoder | Decoder |
|-------|-------|---------|----------|
| **Standard SAE** | $x_t \in \mathbb{R}^d$ | $W_\text{enc} \in \mathbb{R}^{h \times d}$ | $W_\text{dec} \in \mathbb{R}^{d \times h}$ |
| **Temporal Crosscoder** | $[x_t; \ldots; x_{t+T-1}] \in \mathbb{R}^{Td}$ | $W_\text{enc} \in \mathbb{R}^{h \times Td}$ | $W_\text{dec}^{(s)} \in \mathbb{R}^{d \times h}$ per position $s$ |

The crosscoder shares one latent code across all positions in the window, but reconstructs each position separately via its own decoder slice.

## Experiments

| Exp | Data generating process | Expected crosscoder advantage |
|-----|------------------------|-------------------------------|
| 1 | i.i.d. (no temporal correlation) | None — crosscoder ≈ SAE |
| 2 | **Scheme A** — AR(1) magnitude correlation, $\rho=0.9$ | Weak — attenuated by sparsity |
| 3 | **Scheme B** — Gaussian copula support, persistent | Moderate |
| 4 | **Scheme C** — 2-state Markov chain support, $\alpha=0.95, \beta=0.03$ | Strong |


## 0. Install dependencies

In [25]:
# No extra installs needed — sae-lens removed in this version


## 1. Imports & shared helpers

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Any, Callable, NamedTuple
from tqdm.auto import tqdm
from scipy.stats import norm
from torch.distributions import MultivariateNormal
import random
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from transformer_lens.hook_points import HookedRootModule

DEFAULT_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print(f'Device: {DEFAULT_DEVICE}')

Device: cpu


## 2. Toy model — 50 orthogonal features in $\mathbb{R}^{200}$

In [17]:
# ──────────────────────────────────────────────
# Toy model (same as sae_token_embeddings.ipynb)
# ──────────────────────────────────────────────

def orthogonalize(num_vectors: int, vector_len: int, target_cos_sim: float = 0) -> torch.Tensor:
    """Optimise embeddings so all pairwise cosine similarities ≈ target_cos_sim."""
    embeddings = torch.randn(num_vectors, vector_len)
    embeddings = embeddings / embeddings.norm(p=2, dim=1, keepdim=True)
    embeddings.requires_grad_(True)
    optimizer = torch.optim.Adam([embeddings], lr=0.01)
    pbar = tqdm(range(1000), desc='Orthogonalising features', leave=False)
    for _ in pbar:
        optimizer.zero_grad()
        dot = embeddings @ embeddings.T
        diff = dot - target_cos_sim
        diff.fill_diagonal_(0)
        loss = diff.pow(2).sum() + num_vectors * (dot.diag() - 1).pow(2).sum()
        loss.backward()
        optimizer.step()
        pbar.set_description(f'Orthogonalising | loss={loss.item():.3f}')
    with torch.no_grad():
        embeddings = embeddings / embeddings.norm(p=2, dim=1, keepdim=True)
    return embeddings.detach().clone()


class ToyModel(HookedRootModule):
    """Linear map from feature space to representation space."""
    def __init__(self, num_feats: int, hidden_dim: int, target_cos_sim: float = 0):
        super().__init__()
        self.embed = nn.Linear(num_feats, hidden_dim, bias=False)
        embeddings = orthogonalize(num_feats, hidden_dim, target_cos_sim)
        self.embed.weight.data = embeddings.T   # (hidden_dim, num_feats)
        self.setup()

    def forward(self, x: torch.Tensor, **kwargs: Any):
        return self.embed(x)


NUM_FEATS  = 50
HIDDEN_DIM = 200
FEAT_PROB  = 0.05   # sparse: ~2.5 features fire per position
FEAT_MEAN  = 1.0
FEAT_STD   = 0.15

toy_model = ToyModel(NUM_FEATS, HIDDEN_DIM).to(DEFAULT_DEVICE)
toy_model.eval()

feat_probs = torch.ones(NUM_FEATS) * FEAT_PROB
print(f'Toy model: {NUM_FEATS} features | d={HIDDEN_DIM} | p={FEAT_PROB}')

Orthogonalising features:   0%|          | 0/1000 [00:00<?, ?it/s]

Toy model: 50 features | d=200 | p=0.05


## 3. Temporal data generators

Each generator produces sequences of shape `(num_seqs, T, num_feats)` — **before** passing through the toy model. The four schemes follow directly from the problem statement:

- **IID** (Exp 1): $s_{i,t} \sim \text{Bernoulli}(p)$ and $m_{i,t} \sim |\mathcal{N}(\mu, \sigma^2)|$, both independent.
- **Scheme A** (Exp 2): AR(1) magnitude process; support still i.i.d.
- **Scheme B** (Exp 3): Gaussian copula over support indicators; magnitudes i.i.d.
- **Scheme C** (Exp 4): Two-state Markov chain support; magnitudes i.i.d.

In [35]:
# ──────────────────────────────────────────────────────────────────────────
# IID baseline
# ──────────────────────────────────────────────────────────────────────────

def generate_iid_sequences(
    num_seqs: int,
    T: int,
    p: float = FEAT_PROB,
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Returns (num_seqs, T, num_feats) — all i.i.d. Bernoulli gates × |N(mu,sigma)| magnitudes."""
    s = torch.bernoulli(torch.full((num_seqs, T, num_feats), p, device=device))
    m = (torch.randn(num_seqs, T, num_feats, device=device) * sigma + mu).abs()
    return s * m


# ──────────────────────────────────────────────────────────────────────────
# Scheme A — AR(1) magnitude correlation
#
# m_{i,t} = mu + rho*(m_{i,t-1} - mu) + eps_{i,t},  eps ~ N(0, sigma^2*(1-rho^2))
# Support still i.i.d.
# ──────────────────────────────────────────────────────────────────────────

def generate_scheme_a_sequences(
    num_seqs: int,
    T: int,
    rho: float = 0.9,          # AR(1) autocorrelation
    p: float = FEAT_PROB,
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Scheme A: temporally correlated magnitudes via AR(1).  (num_seqs, T, num_feats)"""
    # Innovation std ensures stationary marginal variance = sigma^2
    innov_std = sigma * np.sqrt(1 - rho**2)

    # Initialise from stationary distribution
    m = torch.randn(num_seqs, num_feats, device=device) * sigma + mu   # (N, F)
    m_list = [m]
    for _ in range(T - 1):
        eps = torch.randn_like(m) * innov_std
        m = mu + rho * (m - mu) + eps
        m_list.append(m)

    magnitudes = torch.stack(m_list, dim=1).abs()   # (N, T, F)
    support    = torch.bernoulli(torch.full((num_seqs, T, num_feats), p, device=device))
    return support * magnitudes


# ──────────────────────────────────────────────────────────────────────────
# Scheme B — Gaussian copula over support
#
# Latent z_{i,t} are jointly Gaussian with AR(1) covariance;
# s_{i,t} = 1[z_{i,t} > Phi^{-1}(1-p)]   =>  E[s] = p for all t.
# ──────────────────────────────────────────────────────────────────────────

def _ar1_covariance(T: int, rho: float) -> torch.Tensor:
    """T×T AR(1) correlation matrix."""
    idx = torch.arange(T).float()
    return rho ** torch.abs(idx.unsqueeze(0) - idx.unsqueeze(1))


def generate_scheme_b_sequences(
    num_seqs: int,
    T: int,
    rho: float = 0.85,         # support autocorrelation
    p: float = FEAT_PROB,
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Scheme B: Gaussian copula support.  (num_seqs, T, num_feats)"""
    cov = _ar1_covariance(T, rho).to(device)
    mvn = MultivariateNormal(torch.zeros(T, device=device), covariance_matrix=cov)

    threshold = float(norm.ppf(1 - p))   # Phi^{-1}(1-p)

    # Sample latent: (num_seqs, num_feats, T)  then transpose
    z = mvn.sample((num_seqs, num_feats))          # (N, F, T)
    z = z.permute(0, 2, 1)                          # (N, T, F)
    support = (z > threshold).float()

    magnitudes = (torch.randn(num_seqs, T, num_feats, device=device) * sigma + mu).abs()
    return support * magnitudes


# ──────────────────────────────────────────────────────────────────────────
# Scheme C — 2-state Markov chain support
#
# P(s_t=1 | s_{t-1}=1) = alpha   (stay-on probability)
# P(s_t=1 | s_{t-1}=0) = beta    (turn-on probability)
# Stationary p = beta / (1 - alpha + beta)
# ──────────────────────────────────────────────────────────────────────────

def generate_scheme_c_sequences(
    num_seqs: int,
    T: int,
    alpha: float = 0.95,       # stay-on
    beta:  float = 0.03,       # turn-on
    mu: float = FEAT_MEAN,
    sigma: float = FEAT_STD,
    num_feats: int = NUM_FEATS,
    device: torch.device = DEFAULT_DEVICE,
) -> torch.Tensor:
    """Scheme C: 2-state Markov chain support.  (num_seqs, T, num_feats)"""
    p_stat = beta / (1 - alpha + beta)     # stationary probability

    # Initialise from stationary distribution
    s = torch.bernoulli(torch.full((num_seqs, num_feats), p_stat, device=device))
    support_list = [s.clone()]

    for _ in range(T - 1):
        # Transition probabilities for each cell
        p_next = torch.where(s == 1,
                             torch.full_like(s, alpha),
                             torch.full_like(s, beta))
        s = torch.bernoulli(p_next)
        support_list.append(s.clone())

    support    = torch.stack(support_list, dim=1)                    # (N, T, F)
    magnitudes = (torch.randn(num_seqs, T, num_feats, device=device) * sigma + mu).abs()
    return support * magnitudes


# ──────────────────────────────────────────────────────────────────────────
# Diagnostic: measure empirical autocorrelation at lag 1
# ──────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def empirical_lag1_autocorr(
    feat_seqs: torch.Tensor,   # (N, T, F)
) -> float:
    """Mean lag-1 autocorrelation of activation magnitudes across features."""
    a = feat_seqs            # (N, T, F)
    a_t  = a[:, :-1, :]     # (N, T-1, F)
    a_t1 = a[:,  1:, :]     # (N, T-1, F)

    # Per-feature autocorrelation: average over (N, T-1)
    def corr(x, y):
        x = x - x.mean()
        y = y - y.mean()
        return (x * y).sum() / (x.norm() * y.norm() + 1e-8)

    corrs = []
    for f in range(feat_seqs.shape[-1]):
        xf = a_t[:, :, f].flatten()
        yf = a_t1[:, :, f].flatten()
        corrs.append(corr(xf, yf).item())
    return float(np.mean(corrs))


# Quick sanity check
T_SEQ = 8   # window length used throughout
N_DIAG = 2000

for name, seqs in [
    ('IID',      generate_iid_sequences(N_DIAG, T_SEQ)),
    ('Scheme A', generate_scheme_a_sequences(N_DIAG, T_SEQ, rho=0.9)),
    ('Scheme B', generate_scheme_b_sequences(N_DIAG, T_SEQ, rho=0.85)),
    ('Scheme C', generate_scheme_c_sequences(N_DIAG, T_SEQ, alpha=0.95, beta=0.03)),
]:
    ac = empirical_lag1_autocorr(seqs)
    print(f'{name:12s}  lag-1 autocorr = {ac:.4f}')

IID           lag-1 autocorr = 0.0015
Scheme A      lag-1 autocorr = 0.0024
Scheme B      lag-1 autocorr = 0.5202
Scheme C      lag-1 autocorr = 0.8887


## 4. Temporal Crosscoder architecture (TopK)

The crosscoder takes a window of $T$ consecutive activations and produces a single latent code $u$ shared across the window. Each position gets its own decoder slice.

$$u = \\text{TopK}_k\\bigl(W_\\text{enc}[x_t; \\ldots; x_{t+T-1}] + b_\\text{enc}\\bigr)$$
$$\\hat{x}_{t+s} = W_\\text{dec}^{(s)}\\, u + b_\\text{dec}^{(s)}, \\quad s = 0, \\ldots, T-1$$
$$\\mathcal{L} = \\sum_{s=0}^{T-1} \\|\\hat{x}_{t+s} - x_{t+s}\\|_2^2$$

**TopK replaces ReLU + L1** — exactly $k=11$ latents fire per window. This eliminates the L1 coefficient hyperparameter that caused all previous failures.

$k=11$ rationale: under IID with $p=0.05$, $T=8$, the expected number of unique features appearing at least once is $d_{\\text{sae}}(1-(1-p)^T) \\approx 17$. $k=11$ is slightly below this to keep codes crisp.

In [36]:
# TopK crosscoder: exactly k latents fire per window — no L1 tuning needed.
# TOP_K = 17  (BatchTopK, k per sample — same for SAE and crosscoder)

TOP_K = 3   # latents active per window / per token


class TemporalCrosscoder(nn.Module):
    """
    Temporal crosscoder with TopK activation.
    Encoder:  W_enc ∈ R^{h × (T*d_in)}
    Decoders: W_dec ∈ R^{T × d_in × h}  — one decoder slice per position
    Activation: TopK (k=3) — exactly TOP_K latents active per forward pass
    """
    def __init__(self, d_in: int, d_sae: int, T: int, k: int = TOP_K):
        super().__init__()
        self.d_in  = d_in
        self.d_sae = d_sae
        self.T     = T
        self.k     = k

        self.b_dec = nn.Parameter(torch.zeros(T, d_in))
        self.W_enc = nn.Parameter(torch.empty(d_sae, T * d_in))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(T, d_in, d_sae))

        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data = self.W_dec.data / norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d) → u: (B, h) with exactly k non-zero entries per row."""
        x_centered = x - self.b_dec.unsqueeze(0)
        x_flat     = x_centered.reshape(x.shape[0], -1)
        pre        = x_flat @ self.W_enc.T + self.b_enc
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)
        u = torch.zeros_like(pre)
        u.scatter_(-1, topk_idx, F.relu(topk_vals))
        return u

    def decode(self, u: torch.Tensor) -> torch.Tensor:
        """u: (B, h) → x_hat: (B, T, d)"""
        return torch.einsum('tdh,bh->btd', self.W_dec, u) + self.b_dec.unsqueeze(0)

    def forward(self, x: torch.Tensor):
        """x: (B, T, d) → (recon_loss, x_hat, u)"""
        u          = self.encode(x)
        x_hat      = self.decode(u)
        recon_loss = (x_hat - x).pow(2).sum(dim=-1).mean()
        return recon_loss, x_hat, u

    @torch.no_grad()
    def decoder_directions(self, pos: int = 0) -> torch.Tensor:
        """(d, h) decoder columns for position pos."""
        return self.W_dec[pos]


class TopKSAE(nn.Module):
    """
    Standard SAE with TopK activation — exactly k latents active per token.
    Mirrors the crosscoder's sparsity mechanism; no L1 hyperparameter.
    """
    def __init__(self, d_in: int, d_sae: int, k: int = TOP_K):
        super().__init__()
        self.d_in  = d_in
        self.d_sae = d_sae
        self.k     = k

        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.W_enc = nn.Parameter(torch.empty(d_sae, d_in))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_in, d_sae))

        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        norms = self.W_dec.norm(dim=0, keepdim=True).clamp(min=1e-8)
        self.W_dec.data = self.W_dec.data / norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, d) → u: (B, h) with exactly k non-zero entries per row."""
        x_c = x - self.b_dec.unsqueeze(0)
        pre = x_c @ self.W_enc.T + self.b_enc
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)
        u = torch.zeros_like(pre)
        u.scatter_(-1, topk_idx, F.relu(topk_vals))
        return u

    def decode(self, u: torch.Tensor) -> torch.Tensor:
        """u: (B, h) → x_hat: (B, d)"""
        return u @ self.W_dec.T + self.b_dec.unsqueeze(0)

    def forward(self, x: torch.Tensor):
        """x: (B, d) → (recon_loss, x_hat, u)"""
        u          = self.encode(x)
        x_hat      = self.decode(u)
        recon_loss = (x_hat - x).pow(2).sum(dim=-1).mean()
        return recon_loss, x_hat, u

    @property
    def decoder_directions(self) -> torch.Tensor:
        """(d, h) — decoder columns (already unit-norm)."""
        return self.W_dec


## 5. Training & evaluation helpers

In [ ]:
# ─── Evaluation helpers ────────────────────────────────────────────────────

THRESHOLDS = np.linspace(0, 1, 200)

def cos_sims(mat1: torch.Tensor, mat2: torch.Tensor) -> torch.Tensor:
    """Cosine similarities between columns of mat1 and mat2. Returns (h1, h2)."""
    n1 = mat1 / mat1.norm(dim=0, keepdim=True).clamp(min=1e-8)
    n2 = mat2 / mat2.norm(dim=0, keepdim=True).clamp(min=1e-8)
    return n1.T @ n2


@torch.no_grad()
def feature_recovery_score(
    decoder_directions: torch.Tensor,    # (d, h)
    true_features: torch.Tensor,         # (d, k)
) -> dict:
    sims = cos_sims(decoder_directions, true_features).abs()   # (h, k)
    max_per_true = sims.max(dim=0).values                       # (k,)
    curve = np.array([(max_per_true.cpu().numpy() >= t).mean() for t in THRESHOLDS])
    auc_val = float(np.trapz(curve, THRESHOLDS) / (THRESHOLDS[-1] - THRESHOLDS[0]))
    return {
        'mean_max_cos_sim':  max_per_true.mean().item(),
        'frac_recovered_90': (max_per_true >= 0.9).float().mean().item(),
        'frac_recovered_80': (max_per_true >= 0.8).float().mean().item(),
        'per_feature':       max_per_true.cpu().numpy(),
        'auc':               auc_val,
    }


def print_recovery(name: str, metrics: dict):
    print(f'[{name}]  mean-max cos-sim={metrics["mean_max_cos_sim"]:.3f}  '
          f'| recovered@0.9={metrics["frac_recovered_90"]:.1%}  '
          f'| AUC={metrics["auc"]:.4f}')


# ─── Data iterators ────────────────────────────────────────────────────────────

class ShuffledDataIterator:
    """Randomly sample ONE position from each sequence per call → marginal i.i.d."""
    def __init__(self, model, seq_gen_fn, batch_size, T):
        self.model = model; self.seq_gen_fn = seq_gen_fn
        self.batch_size = batch_size; self.T = T

    @torch.no_grad()
    def next_batch(self):
        feat_seqs = self.seq_gen_fn(self.batch_size, self.T)
        t_idx = torch.randint(0, self.T, (self.batch_size,))
        feats = feat_seqs[torch.arange(self.batch_size), t_idx]
        return self.model(feats)

    def __iter__(self): return self
    def __next__(self): return self.next_batch()


class SequentialDataIterator:
    """Emit T consecutive tokens per sequence in temporal order."""
    def __init__(self, model, seq_gen_fn, batch_size, T):
        self.model = model; self.seq_gen_fn = seq_gen_fn
        self.batch_size = batch_size; self.T = T

    @torch.no_grad()
    def next_batch(self):
        n_seq = max(1, self.batch_size // self.T)
        feat_seqs = self.seq_gen_fn(n_seq, self.T)
        act_seqs  = self.model(feat_seqs)
        flat = act_seqs.reshape(-1, HIDDEN_DIM)
        return flat[:self.batch_size]

    def __iter__(self): return self
    def __next__(self): return self.next_batch()


# ─── Training constants ─────────────────────────────────────────────────────────

TRAIN_STEPS   = 50000   # steps for both SAE and crosscoder
LOG_INTERVAL  = 500       # log every N steps
SAE_BATCH     = 512
CC_BATCH      = 512
EVAL_BATCH    = 2048      # larger batch for stable eval metrics


# ─── SAE training with step-level logging ──────────────────────────────────────

def train_sae(
    seq_gen_fn: Callable,
    toy_model,
    true_features: torch.Tensor,
    mode: str = 'shuffled',          # 'shuffled' | 'sequential'
    d_sae: int = NUM_FEATS,
    k: int = TOP_K,
    n_steps: int = TRAIN_STEPS,
    batch_size: int = SAE_BATCH,
    lr: float = 3e-4,
    device: torch.device = DEFAULT_DEVICE,
):
    """Train TopK SAE and return (model, history). history logged every LOG_INTERVAL steps."""
    assert mode in ('shuffled', 'sequential')
    IterCls  = ShuffledDataIterator if mode == 'shuffled' else SequentialDataIterator
    iterator = IterCls(toy_model, seq_gen_fn, batch_size, T=T_SEQ)

    sae = TopKSAE(d_in=HIDDEN_DIM, d_sae=d_sae, k=k).to(device)
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr, betas=(0.9, 0.999))

    toy_model.eval()
    history = []   # list of {step, loss, auc, recovery_90}
    pbar = tqdm(range(n_steps), desc=f'SAE ({mode})', leave=True)

    for step in pbar:
        # Training step
        x = next(iterator).to(device)
        loss, _, _ = sae(x)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
        optimizer.step()
        sae._normalize_decoder()

        # Periodic evaluation
        if step % LOG_INTERVAL == 0 or step == n_steps - 1:
            with torch.no_grad():
                x_eval = next(iterator).to(device)
                eval_loss, _, _ = sae(x_eval)
                metrics = feature_recovery_score(sae.decoder_directions, true_features)
            history.append({
                'step':        step,
                'loss':        eval_loss.item(),
                'auc':         metrics['auc'],
                'recovery_90': metrics['frac_recovered_90'],
            })
            pbar.set_postfix(loss=f'{eval_loss.item():.4f}',
                             auc=f'{metrics["auc"]:.3f}',
                             rec90=f'{metrics["frac_recovered_90"]:.2f}')

    return sae, history


# ─── Crosscoder training with step-level logging ────────────────────────────────

def train_crosscoder(
    seq_gen_fn: Callable,
    toy_model,
    true_features: torch.Tensor,
    T: int = T_SEQ,
    d_sae: int = NUM_FEATS,
    k: int = TOP_K,
    n_steps: int = TRAIN_STEPS,
    batch_size: int = CC_BATCH,
    lr: float = 3e-4,
    device: torch.device = DEFAULT_DEVICE,
):
    """Train TopK crosscoder and return (model, history)."""
    crosscoder = TemporalCrosscoder(d_in=HIDDEN_DIM, d_sae=d_sae, T=T, k=k).to(device)
    optimizer  = torch.optim.Adam(crosscoder.parameters(), lr=lr, betas=(0.9, 0.999))

    toy_model.eval()
    history = []
    pbar = tqdm(range(n_steps), desc='Crosscoder', leave=True)

    for step in pbar:
        feat_seqs = seq_gen_fn(batch_size, T).to(device)
        with torch.no_grad():
            act_seqs = toy_model(feat_seqs)

        loss, _, u = crosscoder(act_seqs)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(crosscoder.parameters(), 1.0)
        optimizer.step()
        crosscoder._normalize_decoder()

        if step % LOG_INTERVAL == 0 or step == n_steps - 1:
            with torch.no_grad():
                feat_eval = seq_gen_fn(EVAL_BATCH, T).to(device)
                act_eval  = toy_model(feat_eval)
                eval_loss, _, _ = crosscoder(act_eval)
                metrics = feature_recovery_score(
                    crosscoder.decoder_directions(pos=0), true_features
                )
            history.append({
                'step':        step,
                'loss':        eval_loss.item(),
                'auc':         metrics['auc'],
                'recovery_90': metrics['frac_recovered_90'],
            })
            pbar.set_postfix(loss=f'{eval_loss.item():.4f}',
                             auc=f'{metrics["auc"]:.3f}',
                             rec90=f'{metrics["frac_recovered_90"]:.2f}')

    return crosscoder, history


# ─── Plotting helpers ───────────────────────────────────────────────────────────

def plot_recovery_heatmap(decoder_directions, true_features, title,
                           height=420, width=520):
    sims = cos_sims(decoder_directions, true_features)
    fig = go.Figure(go.Heatmap(
        z=sims.detach().cpu().numpy(),
        zmin=-1, zmax=1, colorscale='RdBu',
        colorbar=dict(title='cos sim', dtick=1, tickvals=[-1, 0, 1]),
        hovertemplate='True feature: %{x}<br>Latent: %{y}<br>Cos sim: %{z:.3f}<extra></extra>',
    ))
    fig.update_layout(title=title, xaxis_title='True feature',
                      yaxis_title='Latent', height=height, width=width)
    fig.show()


# Grab true feature directions from toy model: (d, k)
TRUE_FEATURES = toy_model.embed.weight.detach()   # (d, num_feats)
print(f'True feature matrix shape: {TRUE_FEATURES.shape}')

# Container for all results
ALL_RESULTS: list[dict] = []


True feature matrix shape: torch.Size([200, 50])


## 5b. Why the shuffled SAE always gets ~1.0 — covariance analysis

**The marginal argument.** All our temporal processes are *stationary*: the distribution at any single position $t$ is the same Bernoulli×|Normal| regardless of $t$ or of the inter-token correlation structure. When the SAE's `ShuffledDataIterator` picks a random timestep from each sequence, it sees exactly this marginal — an i.i.d. stream identical for IID, Scheme A, B, and C. So near-perfect recovery is expected and correct; it tells us nothing about temporal structure.

**The sequential question.** If we instead feed tokens in their natural temporal order (consecutive within a sequence), the SAE mini-batch has autocorrelated samples. The *effective* independent batch size shrinks by a factor related to the autocorrelation time:
$$
n_{\text{eff}} \approx \frac{n}{1 + 2\sum_{\tau=1}^{\infty} \rho_\tau} \approx \frac{n}{1 + 2\rho/(1-\rho)}
$$
For Scheme C with $\rho \approx 0.84$: $n_{\text{eff}} \approx n/12$. This should slow convergence and potentially degrade recovery — especially for rarely-fired features that only appear in short bursts.

**The covariance matrices below** show the $T \times T$ structure in each scheme and confirm that IID has diagonal structure while Schemes B and C have strong off-diagonal correlations.

In [22]:
# ── T×T Covariance & Correlation matrices per scheme ─────────────────────────

@torch.no_grad()
def compute_TxT_matrices(seq_gen_fn: Callable, N: int = 20_000) -> dict:
    """
    Compute empirical T×T covariance / correlation of the *scalar* activation
    a_{i,t} = s_{i,t} * m_{i,t}, averaged over all k features.
    Also compute tr(Cov(x_t, x_{t'})) — how that covariance propagates into R^d.
    """
    feat_seqs = seq_gen_fn(N, T_SEQ)                        # (N, T, k)
    act_seqs  = toy_model(feat_seqs.to(DEFAULT_DEVICE)).cpu()  # (N, T, d)
    feat_seqs = feat_seqs.cpu()

    k = feat_seqs.shape[-1]
    cov_sum  = torch.zeros(T_SEQ, T_SEQ)
    corr_sum = torch.zeros(T_SEQ, T_SEQ)

    for i in range(k):
        a   = feat_seqs[:, :, i]                            # (N, T)
        a_c = a - a.mean(dim=0)
        C   = (a_c.T @ a_c) / (N - 1)                      # (T, T) cov
        std = C.diag().sqrt().clamp(min=1e-8)
        R   = C / (std.unsqueeze(0) * std.unsqueeze(1))     # (T, T) corr
        cov_sum  += C
        corr_sum += R

    mean_cov  = cov_sum  / k
    mean_corr = corr_sum / k

    # tr(Cov(x_t, x_{t'})): sum of element-wise products over d dims
    act_c   = act_seqs - act_seqs.mean(dim=0)               # (N, T, d)
    rep_cov = torch.zeros(T_SEQ, T_SEQ)
    for t in range(T_SEQ):
        for tp in range(T_SEQ):
            rep_cov[t, tp] = (act_c[:, t, :] * act_c[:, tp, :]).sum(dim=-1).mean().item()

    return dict(cov=mean_cov, corr=mean_corr, rep_cov=rep_cov)


def print_TxT(scheme_name: str, seq_gen_fn: Callable, N: int = 20_000):
    mats  = compute_TxT_matrices(seq_gen_fn, N=N)
    W     = 72
    sep   = '  '

    def fmt_mat(M: torch.Tensor, fmt: str = '{:+.3f}') -> list[str]:
        return [sep.join(fmt.format(v) for v in row.tolist()) for row in M]

    print('\n' + '═' * W)
    print(f'  {scheme_name}')
    print('═' * W)

    print(f'\n  [1] Activation covariance  Cov(a_{{i,t}}, a_{{i,t\'}})  '
          f'(avg over {NUM_FEATS} features, N={N})')
    print(f'  Diagonal = Var(a_i,t).  Off-diagonal > 0 → temporal co-activation.')
    for line in fmt_mat(mats['cov']): print('    ' + line)

    print(f'\n  [2] Activation correlation  Corr(a_{{i,t}}, a_{{i,t\'}})  '
          f'(avg over {NUM_FEATS} features)')
    print(f'  IID → identity.  Scheme C → geometric decay at rate α-β.')
    for line in fmt_mat(mats['corr']): print('    ' + line)

    print(f'\n  [3] Representation covariance  tr(Cov(x_t, x_t\'))')
    print(f'  How activation cov propagates into R^d via the toy model.')
    print(f'  Large off-diagonal = temporal signal exploitable by the crosscoder.')
    for line in fmt_mat(mats['rep_cov']): print('    ' + line)

    c01 = mats['corr'][0, 1].item()
    r01 = mats['rep_cov'][0, 1].item()
    n_eff_ratio = 1 / max(1e-6, 1 + 2 * c01 / max(1e-6, 1 - c01)) if c01 < 1 else 0
    print(f'\n  Lag-1 summary:')
    print(f'    Corr(a_t, a_{{t+1}})            = {c01:+.5f}')
    print(f'    tr(Cov(x_t, x_{{t+1}}))         = {r01:+.5f}')
    print(f'    Effective batch fraction n_eff/n ≈ {n_eff_ratio:.4f}  '
          f'(=1 for IID, <1 for correlated → fewer effective gradient steps)')


# Define seq_gen_fns for schemes before experiments define per-exp versions
_scheme_gen_fns = [
    ('IID',                    lambda N, T: generate_iid_sequences(N, T)),
    ('Scheme A (AR(1) ρ=0.9)', lambda N, T: generate_scheme_a_sequences(N, T, rho=0.9)),
    ('Scheme B (copula ρ=0.85)',lambda N, T: generate_scheme_b_sequences(N, T, rho=0.85)),
    ('Scheme C (Markov α=0.95)',lambda N, T: generate_scheme_c_sequences(N, T, alpha=0.95, beta=0.03)),
]

for name, gfn in _scheme_gen_fns:
    print_TxT(name, gfn, N=20_000)


════════════════════════════════════════════════════════════════════════
  IID
════════════════════════════════════════════════════════════════════════

  [1] Activation covariance  Cov(a_{i,t}, a_{i,t'})  (avg over 50 features, N=20000)
  Diagonal = Var(a_i,t).  Off-diagonal > 0 → temporal co-activation.
    +0.049  +0.000  +0.000  -0.000  +0.000  +0.000  +0.000  -0.000
    +0.000  +0.049  -0.000  -0.000  -0.000  +0.000  +0.000  +0.000
    +0.000  -0.000  +0.049  -0.000  +0.000  -0.000  +0.000  +0.000
    -0.000  -0.000  -0.000  +0.049  -0.000  +0.000  -0.000  +0.000
    +0.000  -0.000  +0.000  -0.000  +0.049  -0.000  -0.000  -0.000
    +0.000  +0.000  -0.000  +0.000  -0.000  +0.048  +0.000  -0.000
    +0.000  +0.000  +0.000  -0.000  -0.000  +0.000  +0.048  +0.000
    -0.000  +0.000  +0.000  +0.000  -0.000  -0.000  +0.000  +0.049

  [2] Activation correlation  Corr(a_{i,t}, a_{i,t'})  (avg over 50 features)
  IID → identity.  Scheme C → geometric decay at rate α-β.
    +1.000  +0.001

---
## Experiment 1 — IID baseline (no temporal correlation)

Both SAE modes and the crosscoder see the same statistical signal; no temporal structure exists.
- **Shuffled SAE**: baseline, should be ~1.0.
- **Sequential SAE**: identical to shuffled for IID (no correlation to disrupt batch diversity).
- **Crosscoder**: should match or slightly lag SAE (extra encoder complexity with no signal to reward it).

> **Data**: $s_{i,t} \sim \text{Bernoulli}(0.05)$, $m_{i,t} \sim |\mathcal{N}(1.0, 0.15^2)|$, all i.i.d.

In [38]:
def iid_seq_gen(n: int, T: int) -> torch.Tensor:
    return generate_iid_sequences(n, T)


In [39]:
print(f'=== Exp 1: IID ===')
print(f'Training SAE (shuffled) for {TRAIN_STEPS:,} steps ...')
sae_exp1_shuf, hist_sae1_shuf = train_sae(
    iid_seq_gen, toy_model, TRUE_FEATURES, mode='shuffled')

print(f'Training SAE (sequential) for {TRAIN_STEPS:,} steps ...')
sae_exp1_seq, hist_sae1_seq = train_sae(
    iid_seq_gen, toy_model, TRUE_FEATURES, mode='sequential')

print(f'Training Crosscoder for {TRAIN_STEPS:,} steps ...')
cc_exp1, hist_cc1 = train_crosscoder(
    iid_seq_gen, toy_model, TRUE_FEATURES)


=== Exp 1: IID ===
Training SAE (shuffled) for 5,000 steps ...


SAE (shuffled):   0%|          | 0/5000 [00:00<?, ?it/s]

Training SAE (sequential) for 5,000 steps ...


SAE (sequential):   0%|          | 0/5000 [00:00<?, ?it/s]

Training Crosscoder for 5,000 steps ...


Crosscoder:   0%|          | 0/5000 [00:00<?, ?it/s]

In [40]:
m_sae1_shuf = feature_recovery_score(sae_exp1_shuf.decoder_directions, TRUE_FEATURES)
m_sae1_seq  = feature_recovery_score(sae_exp1_seq.decoder_directions,  TRUE_FEATURES)
m_cc1       = feature_recovery_score(cc_exp1.decoder_directions(pos=0), TRUE_FEATURES)

print('\n=== Exp 1 results ===')
print_recovery('SAE shuffled  ', m_sae1_shuf)
print_recovery('SAE sequential', m_sae1_seq)
print_recovery('Crosscoder    ', m_cc1)

ALL_RESULTS.append({
    'label': 'IID\n(Exp 1)',
    'sae_shuffled':   m_sae1_shuf,
    'sae_sequential': m_sae1_seq,
    'crosscoder':     m_cc1,
    'hist_sae_shuf':  hist_sae1_shuf,
    'hist_sae_seq':   hist_sae1_seq,
    'hist_cc':        hist_cc1,
})

plot_recovery_heatmap(sae_exp1_shuf.decoder_directions, TRUE_FEATURES,
                      'Exp 1: IID — SAE (shuffled)')
plot_recovery_heatmap(cc_exp1.decoder_directions(pos=0), TRUE_FEATURES,
                      'Exp 1: IID — Crosscoder')



=== Exp 1 results ===
[SAE shuffled  ]  mean-max cos-sim=0.939  | recovered@0.9=100.0%  | AUC=0.9391
[SAE sequential]  mean-max cos-sim=0.941  | recovered@0.9=100.0%  | AUC=0.9410
[Crosscoder    ]  mean-max cos-sim=0.752  | recovered@0.9=70.0%  | AUC=0.7517


In [41]:
# (Exp 1 eval consolidated into exp1-eval cell above)


---
## Experiment 2 — Scheme A: AR(1) magnitude correlation (ρ=0.9)

Magnitudes are temporally smooth but support is still i.i.d. Bernoulli.
As derived in the theory, activation autocorr ≈ γ·ρ ≈ p·ρ = 0.045 — very weak.
- **Shuffled SAE**: ≈ 1.0 (marginal unchanged).
- **Sequential SAE**: ≈ identical (activation autocorr too weak to affect batch diversity).
- **Crosscoder**: slight potential gain.


In [ ]:
RHO_A = 0.9
def scheme_a_seq_gen(n, T): return generate_scheme_a_sequences(n, T, rho=RHO_A)


In [ ]:
print(f'=== Exp 2: Scheme A (ρ={RHO_A}) ===')
print(f'Training SAE (shuffled) ...')
sae_exp2_shuf, hist_sae2_shuf = train_sae(
    scheme_a_seq_gen, toy_model, TRUE_FEATURES, mode='shuffled')

print(f'Training SAE (sequential) ...')
sae_exp2_seq, hist_sae2_seq = train_sae(
    scheme_a_seq_gen, toy_model, TRUE_FEATURES, mode='sequential')

print(f'Training Crosscoder ...')
cc_exp2, hist_cc2 = train_crosscoder(
    scheme_a_seq_gen, toy_model, TRUE_FEATURES)


In [ ]:
m_sae2_shuf = feature_recovery_score(sae_exp2_shuf.decoder_directions, TRUE_FEATURES)
m_sae2_seq  = feature_recovery_score(sae_exp2_seq.decoder_directions,  TRUE_FEATURES)
m_cc2       = feature_recovery_score(cc_exp2.decoder_directions(pos=0), TRUE_FEATURES)

print('\n=== Exp 2 results ===')
print_recovery('SAE shuffled  ', m_sae2_shuf)
print_recovery('SAE sequential', m_sae2_seq)
print_recovery('Crosscoder    ', m_cc2)

ALL_RESULTS.append({
    'label': 'Scheme A\n(Exp 2)',
    'sae_shuffled':   m_sae2_shuf,
    'sae_sequential': m_sae2_seq,
    'crosscoder':     m_cc2,
    'hist_sae_shuf':  hist_sae2_shuf,
    'hist_sae_seq':   hist_sae2_seq,
    'hist_cc':        hist_cc2,
})


In [ ]:
# (Exp 2 eval consolidated above)


---
## Experiment 3 — Scheme B: Gaussian copula support (ρ=0.85)

Support process is correlated; activation lag-1 autocorr ≈ 0.52 (see covariance cell).
- **Shuffled SAE**: ≈ 1.0.
- **Sequential SAE**: effective batch shrinks by ~1/(1+2·0.52/(1-0.52)) ≈ 0.31 → may degrade slightly.
- **Crosscoder**: moderate expected gain.


In [ ]:
RHO_B = 0.85
def scheme_b_seq_gen(n, T): return generate_scheme_b_sequences(n, T, rho=RHO_B)


In [ ]:
print(f'=== Exp 3: Scheme B (ρ={RHO_B}) ===')
print(f'Training SAE (shuffled) ...')
sae_exp3_shuf, hist_sae3_shuf = train_sae(
    scheme_b_seq_gen, toy_model, TRUE_FEATURES, mode='shuffled')

print(f'Training SAE (sequential) ...')
sae_exp3_seq, hist_sae3_seq = train_sae(
    scheme_b_seq_gen, toy_model, TRUE_FEATURES, mode='sequential')

print(f'Training Crosscoder ...')
cc_exp3, hist_cc3 = train_crosscoder(
    scheme_b_seq_gen, toy_model, TRUE_FEATURES)


In [ ]:
m_sae3_shuf = feature_recovery_score(sae_exp3_shuf.decoder_directions, TRUE_FEATURES)
m_sae3_seq  = feature_recovery_score(sae_exp3_seq.decoder_directions,  TRUE_FEATURES)
m_cc3       = feature_recovery_score(cc_exp3.decoder_directions(pos=0), TRUE_FEATURES)

print('\n=== Exp 3 results ===')
print_recovery('SAE shuffled  ', m_sae3_shuf)
print_recovery('SAE sequential', m_sae3_seq)
print_recovery('Crosscoder    ', m_cc3)

ALL_RESULTS.append({
    'label': 'Scheme B\n(Exp 3)',
    'sae_shuffled':   m_sae3_shuf,
    'sae_sequential': m_sae3_seq,
    'crosscoder':     m_cc3,
    'hist_sae_shuf':  hist_sae3_shuf,
    'hist_sae_seq':   hist_sae3_seq,
    'hist_cc':        hist_cc3,
})


In [ ]:
# (Exp 3 eval consolidated above)


---
## Experiment 4 — Scheme C: 2-state Markov chain (α=0.95, β=0.03)

Strongest temporal correlation. Lag-1 autocorr ≈ 0.88. Features stay active for long bursts.
- **Shuffled SAE**: ≈ 1.0.
- **Sequential SAE**: effective batch ≈ n/15 → expected meaningful degradation for rare features.
- **Crosscoder**: largest expected gain — sustained activations provide maximal joint evidence.


In [42]:
ALPHA_C = 0.95; BETA_C = 0.03
def scheme_c_seq_gen(n, T): return generate_scheme_c_sequences(n, T, alpha=ALPHA_C, beta=BETA_C)


In [43]:
print(f'=== Exp 4: Scheme C (α={ALPHA_C}, β={BETA_C}) ===')
print(f'Training SAE (shuffled) ...')
sae_exp4_shuf, hist_sae4_shuf = train_sae(
    scheme_c_seq_gen, toy_model, TRUE_FEATURES, mode='shuffled')

# print(f'Training SAE (sequential) ...')
# sae_exp4_seq, hist_sae4_seq = train_sae(
#     scheme_c_seq_gen, toy_model, TRUE_FEATURES, mode='sequential')

# print(f'Training Crosscoder ...')
# cc_exp4, hist_cc4 = train_crosscoder(
#     scheme_c_seq_gen, toy_model, TRUE_FEATURES)


=== Exp 4: Scheme C (α=0.95, β=0.03) ===
Training SAE (shuffled) ...


SAE (shuffled):   0%|          | 0/5000 [00:00<?, ?it/s]

In [44]:
print(f'Training Crosscoder ...')
cc_exp4, hist_cc4 = train_crosscoder(
    scheme_c_seq_gen, toy_model, TRUE_FEATURES)

Training Crosscoder ...


Crosscoder:   0%|          | 0/5000 [00:00<?, ?it/s]

In [45]:
m_sae4_shuf = feature_recovery_score(sae_exp4_shuf.decoder_directions, TRUE_FEATURES)
m_sae4_seq  = feature_recovery_score(sae_exp4_seq.decoder_directions,  TRUE_FEATURES)
m_cc4       = feature_recovery_score(cc_exp4.decoder_directions(pos=0), TRUE_FEATURES)

print('\n=== Exp 4 results ===')
print_recovery('SAE shuffled  ', m_sae4_shuf)
print_recovery('SAE sequential', m_sae4_seq)
print_recovery('Crosscoder    ', m_cc4)

ALL_RESULTS.append({
    'label': 'Scheme C\n(Exp 4)',
    'sae_shuffled':   m_sae4_shuf,
    'sae_sequential': m_sae4_seq,
    'crosscoder':     m_cc4,
    'hist_sae_shuf':  hist_sae4_shuf,
    'hist_sae_seq':   hist_sae4_seq,
    'hist_cc':        hist_cc4,
})



=== Exp 4 results ===
[SAE shuffled  ]  mean-max cos-sim=0.421  | recovered@0.9=0.0%  | AUC=0.4206
[SAE sequential]  mean-max cos-sim=0.703  | recovered@0.9=30.0%  | AUC=0.7030
[Crosscoder    ]  mean-max cos-sim=0.303  | recovered@0.9=0.0%  | AUC=0.3036


In [46]:
# (Exp 4 eval consolidated above)


---
## Summary — all experiments

We compare feature recovery across all four conditions using three metrics:
1. **Mean max cosine similarity** — averaged over true features.
2. **Recovery @ 0.9** — fraction of true features with a crosscoder latent aligned to ≥ 0.9 cosine similarity.
3. **Recovery @ 0.8** — same with a looser threshold.

In [34]:
# ── Summary table ──────────────────────────────────────────────────────────
print(f'\n{"Exp":<22} {"SAE-shuf AUC":>13} {"SAE-seq AUC":>12} {"XC AUC":>9} {"XC vs shuf":>11}')
print('-' * 72)
for r in ALL_RESULTS:
    label = r['label'].replace('\n', ' ')
    auc_s = r['sae_shuffled']['auc']
    auc_q = r['sae_sequential']['auc']
    auc_c = r['crosscoder']['auc']
    delta = auc_c - auc_s
    flag  = '  <- XC wins' if delta > 0.01 else ('  <- XC loses' if delta < -0.01 else '  ~')
    print(f'{label:<22} {auc_s:>13.4f} {auc_q:>12.4f} {auc_c:>9.4f} {delta:>+11.4f}{flag}')


# ── Recovery curves (final) ──────────────────────────────────────────────────
def recovery_curve(per_feature, thresholds):
    return np.array([(per_feature >= t).mean() for t in thresholds])

model_styles = {
    'sae_shuffled':   ('SAE - shuffled',    'steelblue',      'solid'),
    'sae_sequential': ('SAE - sequential',  'cornflowerblue', 'dot'),
    'crosscoder':     ('Crosscoder (TopK)', 'firebrick',      'solid'),
}

fig = make_subplots(
    rows=1, cols=len(ALL_RESULTS),
    subplot_titles=[r['label'].replace('\n', '<br>') for r in ALL_RESULTS],
    shared_yaxes=True,
)
for col, r in enumerate(ALL_RESULTS, start=1):
    for key, (name, color, dash) in model_styles.items():
        curve = recovery_curve(r[key]['per_feature'], THRESHOLDS)
        fig.add_trace(go.Scatter(
            x=THRESHOLDS, y=curve, name=name,
            line=dict(color=color, dash=dash, width=2),
            showlegend=(col == 1), legendgroup=key,
        ), row=1, col=col)

fig.update_layout(
    title='Feature recovery curves — fraction recovered vs cosine similarity threshold',
    height=420, width=1100,
    legend=dict(x=0.01, y=-0.2, orientation='h'),
)
fig.update_xaxes(title_text='cos-sim threshold', range=[0, 1])
fig.update_yaxes(title_text='Fraction recovered', range=[0, 1.05], col=1)
fig.show()



Exp                     SAE-shuf AUC  SAE-seq AUC    XC AUC  XC vs shuf
------------------------------------------------------------------------
IID (Exp 1)                   0.4544       0.4745    0.5737     +0.1193  <- XC wins
Scheme C (Exp 4)              0.7685       0.7030    0.3402     -0.4283  <- XC loses


In [ ]:
# ── Combined training curves: AUC, Recovery@0.9, Loss — all experiments ──────
#
# 4 experiments × 3 models × 3 metrics = 12 subplot panels (4 cols × 3 rows)

METRICS = [
    ('auc',         'AUC',              [0, 1.05]),
    ('recovery_90', 'Recovery @ 0.9',   [0, 1.05]),
    ('loss',        'Reconstruction Loss', None),
]

MODEL_CURVES = [
    ('hist_sae_shuf', 'SAE — shuffled',   'steelblue',      'solid'),
    ('hist_sae_seq',  'SAE — sequential', 'cornflowerblue', 'dot'),
    ('hist_cc',       'Crosscoder',       'firebrick',       'solid'),
]

n_exp = len(ALL_RESULTS)
n_met = len(METRICS)

fig = make_subplots(
    rows=n_met, cols=n_exp,
    subplot_titles=[
        f'{r["label"].replace(chr(10), " ")}' for r in ALL_RESULTS
    ] * 1 + [''] * (n_exp * (n_met - 1)),   # titles only on top row
    shared_xaxes='columns',
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
)

# Fix: subplot_titles only takes one per subplot; rebuild manually
fig = make_subplots(
    rows=n_met, cols=n_exp,
    shared_xaxes='columns',
    vertical_spacing=0.10,
    horizontal_spacing=0.07,
)

for col, r in enumerate(ALL_RESULTS, start=1):
    exp_label = r['label'].replace('\n', ' ')
    for row, (metric_key, metric_name, y_range) in enumerate(METRICS, start=1):
        for hist_key, model_name, color, dash in MODEL_CURVES:
            hist = r[hist_key]
            steps = [h['step'] for h in hist]
            vals  = [h[metric_key] for h in hist]
            fig.add_trace(
                go.Scatter(
                    x=steps, y=vals,
                    name=model_name,
                    line=dict(color=color, dash=dash, width=1.8),
                    showlegend=(row == 1 and col == 1),
                    legendgroup=hist_key,
                    hovertemplate=f'{model_name}<br>step=%{{x}}<br>{metric_name}=%{{y:.4f}}<extra></extra>',
                ),
                row=row, col=col,
            )
        # Y-axis label (left column only) and optional range
        yaxis_kw = dict(title_text=metric_name) if col == 1 else {}
        if y_range:
            yaxis_kw['range'] = y_range
        if yaxis_kw:
            axis_id = '' if (row == 1 and col == 1) else f'{(row-1)*n_exp + col}'
            fig.update_yaxes(row=row, col=col, **yaxis_kw)

        # X-axis label (bottom row only)
        if row == n_met:
            fig.update_xaxes(title_text='Training step', row=row, col=col)

        # Experiment label as annotation at top of each column (first row)
        if row == 1:
            fig.add_annotation(
                text=f'<b>{exp_label}</b>',
                xref=f'x{(col) if col > 1 else ""}', yref='paper',
                x=0.5, y=1.02, xanchor='center', yanchor='bottom',
                showarrow=False, font=dict(size=11),
            )

fig.update_layout(
    title=(
        f'Training curves — TopK SAE & Crosscoder (k={TOP_K}, 100k steps each)<br>'
        f'<sup>Logged every {LOG_INTERVAL} steps | 3 metrics × 4 experiments</sup>'
    ),
    height=780, width=1200,
    legend=dict(x=0.0, y=-0.08, orientation='h', bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(t=100),
)
fig.show()


# ── Per-experiment training-curve summary (larger, one figure per exp) ─────────
for r in ALL_RESULTS:
    exp_label = r['label'].replace('\n', ' ')
    fig2 = make_subplots(rows=1, cols=3,
                         subplot_titles=['AUC', 'Recovery @ 0.9 cos-sim', 'Loss'],
                         shared_xaxes=True)
    for hist_key, model_name, color, dash in MODEL_CURVES:
        hist  = r[hist_key]
        steps = [h['step'] for h in hist]
        for col_i, (metric_key, _, _) in enumerate(METRICS, start=1):
            vals = [h[metric_key] for h in hist]
            fig2.add_trace(
                go.Scatter(x=steps, y=vals, name=model_name,
                           line=dict(color=color, dash=dash, width=2),
                           showlegend=(col_i == 1), legendgroup=hist_key),
                row=1, col=col_i,
            )
    fig2.update_layout(
        title=f'Training curves — {exp_label}',
        height=380, width=1050,
        legend=dict(x=0.01, y=-0.25, orientation='h'),
    )
    fig2.update_xaxes(title_text='Training step')
    fig2.update_yaxes(range=[0, 1.05], col=1)
    fig2.update_yaxes(range=[0, 1.05], col=2)
    fig2.show()


---
## Discussion

### What we expect to see

| Experiment | SAE | Crosscoder | Reason |
|-----------|-----|------------|--------|
| IID | baseline | ≈ SAE | No temporal signal exists |
| Scheme A (AR(1) mags) | baseline | slight gain | Weak signal due to sparsity attenuation $\gamma \leq p = 0.05$ |
| Scheme B (copula support) | baseline | moderate gain | Correlated support gives crosscoder more co-occurrence evidence |
| Scheme C (Markov support) | baseline | largest gain | Strong, sustained activations make joint encoding much easier |

### Intuition for _why_ the crosscoder helps

A standard SAE sees each position independently. Under IID data, this is no loss of information. But when the support process is correlated, observing $x_t$ gives information about $x_{t+1}$: if feature $i$ is active now, it's likely still active next step. The crosscoder's shared encoder sees the whole window $[x_t, \ldots, x_{t+T-1}]$ simultaneously and can use this joint evidence to more confidently identify which features are active, effectively "denoising" through temporal pooling.

### Limitations of this toy model

- We use **ReLU** (not TopK or JumpReLU) for simplicity; a TopK crosscoder would enforce exact sparsity and may perform differently.
- The crosscoder training loop is custom (not via sae-lens) and may not be perfectly comparable in terms of hyperparameter tuning.
- Real LLM activations have many more features and more complex temporal structure — but this toy setup establishes the principle.

### Next steps

- **Sweep $T$**: does a longer window continue to help, or does the marginal benefit saturate?
- **Sweep $\rho / \alpha$**: map out the phase transition where the crosscoder begins outperforming the SAE.
- **Superposition**: apply a bottleneck $\mathbf{W}_\text{model} \in \mathbb{R}^{d' \times d}$ with $d' < k$ and see whether temporal information also aids feature recovery under superposition.
- **Real models**: apply temporal crosscoders to GPT-2 or Llama activations and compare feature interpretability scores.